# Rare Disease Phenotype Matcher

Describe a set of symptoms in plain English → get ranked candidate rare diseases, matched against the **Human Phenotype Ontology (HPO)** — the real clinical vocabulary used by geneticists — with a local, free LLM explaining the reasoning.

**100% free, no API keys, no signups:**
- **Dataset**: HPO ontology + disease annotations — public, from the Monarch Initiative / Jackson Lab
- **Embeddings**: `all-MiniLM-L6-v2` — free, runs locally
- **LLM**: `Qwen2.5-0.5B-Instruct` — free, open-weight, runs on CPU (no GPU needed)

**How it's different from a typical symptom checker:** instead of just asking an LLM "what disease is this?" (which hallucinates), this pipeline:
1. Semantically matches your symptom text to standardized HPO phenotype terms
2. Scores candidate diseases using **information content weighting** — a symptom shared by 500 diseases barely counts, but one seen in only 3 diseases is a strong signal (this is how real clinical phenotype-matching tools like Phenomizer work)
3. Only then uses the LLM to explain the ranked shortlist — it never freely diagnoses, it narrates evidence you can verify

**Important:** this is a research/educational triage aid, not a diagnostic tool. It does not replace a clinical geneticist. Rare disease diagnosis is complex and this is meant to illustrate phenotype-matching techniques, not to be used for real medical decisions.

## 1. Install dependencies

**Important:** after this cell finishes, if Colab shows a "RESTART SESSION" button/prompt, click it, then just re-run this cell once more and continue to cell 2 — don't skip ahead. A restart wipes all variables, which is the #1 cause of `NameError` in this notebook.

In [ ]:
!pip install -q sentence-transformers transformers torch requests pandas accelerate

## 2. Download and parse the HPO dataset (public, free)

- `hp.obo`: the ontology itself — ~18,000 phenotype terms with names/definitions
- `phenotype.hpoa`: links phenotype terms to ~11,000+ rare diseases

Source: [Human Phenotype Ontology](https://hpo.jax.org/), maintained by the Monarch Initiative.

Download + parsing are combined into one cell on purpose — if you ever hit a `NameError` about `hpo_terms`, `disease_to_hpo`, etc. being undefined, just re-run this cell (it's safe to re-run; it skips re-downloading if the files already exist).

In [ ]:
import os
import requests
import pandas as pd
from collections import defaultdict

HPO_OBO_URL = "https://purl.obolibrary.org/obo/hp.obo"
HPO_ANNOTATIONS_URL = "https://purl.obolibrary.org/obo/hp/hpoa/phenotype.hpoa"

def download_if_missing(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print(f"  {path} already downloaded, skipping")
        return
    print(f"Downloading {path}...")
    r = requests.get(url)
    r.raise_for_status()
    with open(path, "wb") as f:
        f.write(r.content)
    print(f"  saved {len(r.content) / 1e6:.1f} MB")

download_if_missing(HPO_OBO_URL, "hp.obo")
download_if_missing(HPO_ANNOTATIONS_URL, "phenotype.hpoa")


def parse_hpo_obo(path):
    """Minimal .obo parser — extracts id and name for each [Term] block.
    No external obo library needed, keeps dependencies light."""
    terms = {}
    current_id, current_name = None, None
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line == "[Term]":
                current_id, current_name = None, None
            elif line.startswith("id: HP:"):
                current_id = line.split("id: ")[1]
            elif line.startswith("name:") and current_id:
                current_name = line.split("name: ")[1]
                terms[current_id] = current_name
    return terms

hpo_terms = parse_hpo_obo("hp.obo")
print(f"Parsed {len(hpo_terms)} HPO terms")
print("Example:", list(hpo_terms.items())[:3])

hpoa = pd.read_csv("phenotype.hpoa", sep="\t", comment="#", low_memory=False)
print(f"{len(hpoa)} annotation rows, {hpoa['database_id'].nunique()} unique diseases")

# disease_id -> set of HPO term ids
disease_to_hpo = defaultdict(set)
# disease_id -> human-readable name
disease_names = {}
# hpo_id -> set of diseases that have it (needed for information-content weighting)
hpo_to_diseases = defaultdict(set)

for _, row in hpoa.iterrows():
    disease_id = row["database_id"]
    hpo_id = row["hpo_id"]
    disease_names[disease_id] = row["disease_name"]
    disease_to_hpo[disease_id].add(hpo_id)
    hpo_to_diseases[hpo_id].add(disease_id)

print(f"{len(disease_to_hpo)} diseases with phenotype annotations")
print("Setup complete — hpo_terms, disease_to_hpo, disease_names, hpo_to_diseases are all ready.")

## 3. Embed all HPO term names

This lets us semantically match free-text symptoms ("floppy baby") to the correct clinical term ("Hypotonia") even when the wording doesn't match exactly.

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

# Safety check: if you hit a NameError here, it means the runtime was
# restarted after cell 2 ran (common in Colab after installing packages).
# Just scroll up and re-run cell 2, then come back and re-run this cell.
assert "hpo_terms" in dir(), "hpo_terms is missing — re-run cell 2 (download + parse) first."

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

hpo_ids = list(hpo_terms.keys())
hpo_names = [hpo_terms[i] for i in hpo_ids]

print(f"Embedding {len(hpo_names)} HPO term names (this takes a minute)...")
hpo_embeddings = embed_model.encode(hpo_names, show_progress_bar=True, batch_size=64, convert_to_tensor=True)
print("Done.")

## 4. Match free-text symptoms to HPO terms

Splits the input into candidate phrases, embeds each, and finds the closest HPO term(s) above a similarity threshold.

In [ ]:
import re

def split_symptoms(text):
    """Naive but effective: split on commas, semicolons, and ' and '."""
    parts = re.split(r",|;| and |\. ", text)
    return [p.strip() for p in parts if len(p.strip()) > 2]

def match_phenotypes(text, top_k=1, min_similarity=0.45):
    phrases = split_symptoms(text)
    matched = []
    phrase_embeddings = embed_model.encode(phrases, convert_to_tensor=True)

    for phrase, phrase_emb in zip(phrases, phrase_embeddings):
        sims = util.cos_sim(phrase_emb, hpo_embeddings)[0]
        best_idx = int(torch.argmax(sims))
        best_score = float(sims[best_idx])
        if best_score >= min_similarity:
            matched.append({
                "input_phrase": phrase,
                "hpo_id": hpo_ids[best_idx],
                "hpo_name": hpo_names[best_idx],
                "similarity": round(best_score, 3),
            })
    return matched

# quick test
test_matches = match_phenotypes("floppy baby, delayed speech, seizures, unusually flexible joints")
for m in test_matches:
    print(f"  {m['input_phrase']!r} -> {m['hpo_name']} ({m['hpo_id']}, sim={m['similarity']})")

## 5. Rank candidate diseases (information-content weighted)

A symptom present in thousands of diseases (e.g. "fatigue") is weak evidence. A symptom present in only a handful of diseases is strong evidence. We weight each matched phenotype by `1 / (diseases with that term)` — rarer terms score higher, similar to real clinical phenotype-matching tools like Phenomizer.

In [ ]:
import math

def rank_diseases(matched_terms, top_n=10):
    matched_hpo_ids = [m["hpo_id"] for m in matched_terms]
    scores = defaultdict(float)

    for hpo_id in matched_hpo_ids:
        diseases_with_term = hpo_to_diseases.get(hpo_id, set())
        if not diseases_with_term:
            continue
        # information content: rarer term = higher weight
        weight = math.log(len(disease_to_hpo) / len(diseases_with_term) + 1)
        for disease_id in diseases_with_term:
            scores[disease_id] += weight

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

    results = []
    for disease_id, score in ranked:
        overlap = disease_to_hpo[disease_id].intersection(matched_hpo_ids)
        matched_names = [hpo_terms.get(h, h) for h in overlap]
        results.append({
            "disease_id": disease_id,
            "disease_name": disease_names.get(disease_id, "Unknown"),
            "score": round(score, 2),
            "matched_terms": matched_names,
        })
    return results

test_ranking = rank_diseases(test_matches)
for r in test_ranking[:5]:
    print(f"  {r['score']:.2f}  {r['disease_name']} ({r['disease_id']}) — matched: {r['matched_terms']}")

## 6. Load a free local LLM to explain the results

`Qwen2.5-0.5B-Instruct` is open-weight, ungated (no HF token needed), and small enough to run on CPU. It only explains the shortlist you already computed — it never generates a diagnosis from scratch, which keeps it grounded in your matched evidence.

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
)

def explain_results(symptom_text, matched_terms, ranked_diseases, n=3):
    matches_str = "\n".join(f"- {m['input_phrase']} -> {m['hpo_name']}" for m in matched_terms)
    diseases_str = "\n".join(
        f"{i+1}. {d['disease_name']} (score {d['score']}, matched: {', '.join(d['matched_terms'])})"
        for i, d in enumerate(ranked_diseases[:n])
    )

    prompt = f"""You are explaining the output of a phenotype-matching algorithm, not making a diagnosis.

Original symptom description: {symptom_text}

Symptoms matched to standardized clinical terms:
{matches_str}

Top candidate diseases ranked by phenotype overlap:
{diseases_str}

Write a short (3-4 sentence) plain-English summary of why these diseases were ranked highly, referencing the matched symptoms. End with one sentence reminding the reader this is a research tool, not a diagnosis, and a doctor or genetic counselor should be consulted."""

    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=220, do_sample=False)
    return output[0]["generated_text"][-1]["content"]

explanation = explain_results("floppy baby, delayed speech, seizures, unusually flexible joints", test_matches, test_ranking)
print(explanation)

## 7. Full pipeline — try your own symptom description

In [ ]:
def diagnose(symptom_text, top_n_diseases=5):
    matched = match_phenotypes(symptom_text)
    if not matched:
        print("No symptoms matched confidently to HPO terms. Try rephrasing with more specific clinical language.")
        return

    ranked = rank_diseases(matched, top_n=top_n_diseases)

    print("Matched phenotypes:")
    for m in matched:
        print(f"  - {m['input_phrase']!r} -> {m['hpo_name']} (similarity {m['similarity']})")

    print("\nTop candidate diseases:")
    for r in ranked:
        print(f"  {r['score']:.2f}  {r['disease_name']} — matched: {', '.join(r['matched_terms'])}")

    print("\nExplanation:")
    print(explain_results(symptom_text, matched, ranked))

# Change this to any symptom description you want to test
diagnose("recurrent bone fractures, blue-tinted sclera, hearing loss, short stature")

## Ideas to extend this

- **Better symptom extraction**: replace the naive comma-split with the local LLM extracting phenotype phrases first, before matching
- **Web UI**: wrap `diagnose()` in a small Streamlit or Gradio app for a shareable demo
- **Bigger model**: swap `Qwen2.5-0.5B-Instruct` for `Qwen2.5-1.5B-Instruct` or `Phi-3-mini` if you have GPU access, for better explanations
- **Evaluation**: HPO ships known disease-phenotype pairs — hold some out and measure whether the true disease lands in your top-10 ranked list
- **Add inheritance/frequency data**: `phenotype.hpoa` includes onset and frequency columns you're not using yet — factor those into scoring

## Why this is a solid GitHub project

- Uses a **real clinical ontology** or, not a toy dataset — this is the same terminology system used in EHRs, genetic testing labs, and PubMed indexing
- The information-content weighting is a legitimate technique from clinical bioinformatics, not just "ask the LLM"
- Fully reproducible and free — anyone can clone it and run it with zero cost
- Clear path to a portfolio-worthy extension (Streamlit demo, evaluation benchmark, etc.)